# ME-Model Custom Validation

This notebook demonstrates how to run **custom parametric validations** on an ME-Model
using the OBI-One validation workflow and BlueCelluLab's parametric validation framework.

## Architecture

- **BlueCelluLab** provides the parametric validation primitives: `Protocol`, `Measurement`, `Criterion`
- **OBI-One** provides the workflow orchestration: setup, run, register
- **entitysdk** persists results as `ValidationResult` entities

## How to add a custom validation

Compose three things:
1. A **Protocol** — how to stimulate the cell (e.g., step at 150% rheobase)
2. A **Measurement** — what to extract (e.g., spike count, AP amplitude)
3. A **Criterion** — pass/fail logic (e.g., value > 0, value in range)

Then wrap them in a `ParametricValidation` and pass to the workflow.

### Important: Import Order

NEURON mechanisms must be compiled **before** importing `bluecellulab`.
This notebook follows the required order:
1. Authenticate and download the model
2. Compile mechanisms with `nrnivmodl`
3. **Then** import bluecellulab and define custom validations

## 1. Authentication and Setup

In [ ]:
from obi_auth import get_token
from entitysdk.client import Client
from entitysdk.models import MEModel
from entitysdk.downloaders.memodel import download_memodel
from obi_notebook.get_environment import get_environment
from obi_notebook.get_projects import get_projects

token = get_token(environment=get_environment(), auth_mode="daf")
project_context = get_projects(token)
client = Client(
    project_context=project_context,
    environment=get_environment(),
    token_manager=token,
)

## 2. Select the ME-Model to validate

In [ ]:
# Set your ME-Model ID here, or leave None to select interactively
memodel_id = None  # e.g. "12a65005-a035-4a36-afe0-605de815861d"

if not memodel_id:
    from obi_notebook.get_entities import get_entities
    memodel_ids = get_entities(
        "me-model", token, [],
        project_context=project_context,
        multi_select=False,
        page_size=100,
    )
    memodel_id = memodel_ids[0] if memodel_ids else None

assert memodel_id is not None, "Please provide an ME-Model ID."
print(f"Selected ME-Model: {memodel_id}")

## 3. Download Model and Compile Mechanisms

**This must happen BEFORE importing bluecellulab.**
NEURON loads mechanisms at initialization time, so compiled `.mod` files must exist first.

In [ ]:
import os
import subprocess
from pathlib import Path

output_dir = Path("./validation_output").resolve()
output_dir.mkdir(parents=True, exist_ok=True)

# Fetch metadata and download
memodel = client.get_entity(entity_type=MEModel, entity_id=memodel_id)
downloaded = download_memodel(client, memodel=memodel, output_dir=str(output_dir))

print(f"ME-Model: {memodel.name}")
print(f"HOC: {downloaded.hoc_path}")
print(f"Morphology: {downloaded.morphology_path}")
print(f"Mechanisms: {downloaded.mechanisms_dir}")

In [ ]:
# Compile mechanisms
mechanisms_dir = Path(downloaded.mechanisms_dir).resolve()
result = subprocess.run(
    ["nrnivmodl", str(mechanisms_dir)],
    capture_output=True,
    text=True,
    cwd=str(output_dir),
)
if result.returncode != 0:
    print(f"ERROR: nrnivmodl failed:\n{result.stdout}\n{result.stderr}")
else:
    print("Mechanisms compiled successfully.")

# Change to output dir so NEURON finds compiled mechanisms
os.chdir(output_dir)
print(f"Working directory: {os.getcwd()}")

## 4. Import BlueCelluLab and Create Cell

Now that mechanisms are compiled and we're in the right directory, it's safe to import bluecellulab.

In [ ]:
from bluecellulab.cell.core import Cell
from bluecellulab.circuit.circuit_access import EmodelProperties
from bluecellulab.simulation.neuron_globals import set_neuron_globals
from bluecellulab.tools import calculate_rheobase

set_neuron_globals(temperature=34.0, v_init=-80.0)

# Get calibration data if available
calibration = getattr(memodel, 'calibration_result', None)
holding_current = calibration.holding_current if calibration else 0.0
threshold_current = calibration.threshold_current if calibration else None

emodel_properties = EmodelProperties(
    threshold_current=threshold_current or 0.1,
    holding_current=holding_current,
    AIS_scaler=1.0,
)

cell = Cell(
    template_path=downloaded.hoc_path,
    morphology_path=downloaded.morphology_path,
    template_format="v6",
    emodel_properties=emodel_properties,
)
print(f"Cell created: {cell}")

# Compute rheobase
if threshold_current:
    rheobase = threshold_current
    print(f"Using calibration threshold: {rheobase:.4f} nA")
else:
    rheobase = calculate_rheobase(cell=cell, section="soma[0]", segx=0.5, threshold_voltage=-40.0)
    print(f"Computed rheobase: {rheobase:.4f} nA")

## 5. Define Custom Validations

Now we can import the parametric validation framework and define custom tests.
Each validation is a composition of Protocol + Measurement + Criterion.

In [ ]:
from bluecellulab.validation import (
    EfelMeasurement,
    GreaterThan,
    ParametricValidation,
    SequenceProtocol,
    StepProtocol,
)

# Default OBI preset: spiking at 130% rheobase
spiking_check = ParametricValidation(
    validation_name="Simulatable Neuron Spiking Validation",
    protocol=StepProtocol(threshold_percentage=130.0),
    measurement=EfelMeasurement(feature_name="Spikecount"),
    criterion=GreaterThan(threshold=0),
    figure_filename="spiking_validation.pdf",
)

# Custom 1: Check that the neuron produces at least 3 spikes at 200% rheobase
high_freq_spiking = ParametricValidation(
    validation_name="High-Frequency Spiking Check",
    protocol=StepProtocol(threshold_percentage=200.0),
    measurement=EfelMeasurement(feature_name="Spikecount"),
    criterion=GreaterThan(threshold=3),
    figure_filename="high_freq_spiking.pdf",
)

# Custom 2: Check that AP amplitude is above 50 mV at 130% rheobase
ap_amplitude_check = ParametricValidation(
    validation_name="AP Amplitude Validation",
    protocol=StepProtocol(threshold_percentage=130.0),
    measurement=EfelMeasurement(feature_name="AP_amplitude"),
    criterion=GreaterThan(threshold=50.0),
    figure_filename="ap_amplitude_validation.pdf",
)

# Custom 3: Thalamic rebound burst validation
# Hyperpolarize for 500ms to de-inactivate T-type Ca channels, then release
# and check for rebound bursting spikes.
rebound_burst = ParametricValidation(
    validation_name="Thalamic Rebound Burst Validation",
    protocol=SequenceProtocol(
        phases=[
            (500.0, -200.0),  # 500ms hyperpolarizing step
            (500.0, 0.0),     # 500ms release — expect rebound burst
        ],
        pre_delay=250.0,
        post_delay=250.0,
    ),
    measurement=EfelMeasurement(feature_name="Spikecount"),
    criterion=GreaterThan(threshold=0),
    figure_filename="rebound_burst.pdf",
)

all_tests = [spiking_check, high_freq_spiking, ap_amplitude_check, rebound_burst]
print(f"Defined {len(all_tests)} validations.")

## 6. Run Validations

Execute all tests against the cell.

In [ ]:
# Prepare figure output directory
cell_name = memodel.name or memodel_id
fig_dir = output_dir / cell_name
fig_dir.mkdir(parents=True, exist_ok=True)

# Run all validations
outcomes = []
for test in all_tests:
    outcome = test.run(cell.template_params, rheobase, fig_dir)
    outcomes.append(outcome)
    status = "PASS" if outcome.passed else "FAIL"
    print(f"[{status}] {outcome.name}")
    print(f"        {outcome.details}")

In [ ]:
# Summary
print("\n" + "=" * 60)
print("VALIDATION SUMMARY")
print("=" * 60)
for outcome in outcomes:
    status = "PASS" if outcome.passed else "FAIL"
    print(f"  [{status}] {outcome.name}")
print("=" * 60)
total = sum(1 for o in outcomes if o.passed)
print(f"  {total}/{len(outcomes)} passed")

## 7. Register Results on the Platform

Register each outcome as a `ValidationResult` entity, with figures and details as assets.
Deduplication is built-in: existing results are skipped.

In [ ]:
from obi_one.scientific.validations.registration import register_outcomes

registered = register_outcomes(
    client=client,
    outcomes=outcomes,
    validated_entity_id=memodel_id,
    out_dir=fig_dir,
    skip_if_exists=True,
)

for r in registered:
    if r.skipped:
        print(f"  [SKIPPED] {r.outcome.name} (already exists)")
    elif r.entity_id:
        print(f"  [REGISTERED] {r.outcome.name} (id={r.entity_id})")
    else:
        print(f"  [FAILED] {r.outcome.name}")

## 8. Adding Your Own Custom Validation

### Simple case: compose building blocks

For validations that follow the pattern *stimulate → extract eFEL feature → check threshold*:

```python
my_validation = ParametricValidation(
    validation_name="My Custom Check",
    protocol=StepProtocol(threshold_percentage=150.0),
    measurement=EfelMeasurement(feature_name="AP_rise_time"),
    criterion=GreaterThan(threshold=0.5),
    figure_filename="my_custom_check.pdf",
)
```

For multi-phase protocols (e.g. rebound burst), use `SequenceProtocol`:

```python
my_sequence = ParametricValidation(
    validation_name="My Sequence Check",
    protocol=SequenceProtocol(
        phases=[(500.0, -150.0), (300.0, 0.0)],
        pre_delay=250.0,
        post_delay=250.0,
    ),
    measurement=EfelMeasurement(feature_name="Spikecount"),
    criterion=GreaterThan(threshold=0),
)
```

### Advanced case: subclass ValidationTest

For validations that need multi-location recording, curve fitting, or custom analysis,
subclass `ValidationTest` directly and write your own logic:

```python
from pathlib import Path
from bluecellulab.validation.base import ValidationTest, ValidationOutcome
from bluecellulab.validation.plotting import plot_trace
from bluecellulab.analysis.inject_sequence import run_stimulus
from bluecellulab.stimulus.factory import StimulusFactory
import numpy as np

class RMPStabilityValidation(ValidationTest):
    """Check that resting membrane potential is stable (low variance)."""
    name = "RMP Stability Check"

    def run(self, template_params, rheobase, out_dir):
        out_dir = Path(out_dir)
        # Run with zero current to observe resting state
        stim_factory = StimulusFactory(dt=1.0)
        stimulus = stim_factory.step(pre_delay=100, duration=1000, post_delay=100, amplitude=0.0)
        recording = run_stimulus(template_params, stimulus, "soma[0]", 0.5, add_hypamp=True)

        # Measure voltage variance during the stimulus period
        mask = (recording.time >= 100) & (recording.time <= 1100)
        voltage_std = float(np.std(recording.voltage[mask]))
        passed = voltage_std < 1.0  # less than 1 mV std

        fig_path = plot_trace(recording, out_dir, "rmp_stability.pdf", "RMP Stability")

        return ValidationOutcome(
            name=self.name,
            passed=passed,
            details=f"Voltage std = {voltage_std:.3f} mV (threshold: < 1.0 mV)",
            figures=[fig_path],
        )
```

Both approaches produce a `ValidationOutcome` that can be registered on the platform.

### Available building blocks

**Protocols:**
- `StepProtocol(threshold_percentage=...)` — Single step at X% of rheobase
- `SequenceProtocol(phases=[(duration, pct), ...])` — Multi-phase stimulus sequence

**Measurements:**
- `EfelMeasurement(feature_name=...)` — any eFEL feature (Spikecount, AP_amplitude, etc.)

**Criteria:**
- `GreaterThan(threshold=...)` — value must exceed threshold

More measurements and criteria will be added as the framework evolves.